# Part 3: Regression

**Course:** 2026 KMITL Data Analytics

This notebook teaches regression for continuous numbers: a straight line,
several features, error metrics, gradient descent and polynomial curves.

## Learning objectives

By the end of this notebook you can:

1. Explain what regression is and why its target is a continuous number.
2. Name features and target in a house-price dataset.
3. Fit simple linear regression by hand with the OLS formulas.
4. Explain weight (slope), bias (intercept) and their units.
5. Compute predictions, errors and residuals with one sign rule.
6. Fit multiple linear regression with scikit-learn.
7. Explain error, loss and cost.
8. Compute MAE, MSE, RMSE and R-squared by hand and with code.
9. Choose a metric and explain negative R-squared and constant targets.
10. Describe why OLS minimises squared residuals.
11. Run gradient descent and read a cost-history chart.
12. Compare a straight line with a fixed-degree polynomial model.

## Required imports

`numpy` for numbers, `pandas` for the table, `matplotlib` for charts, and
scikit-learn for models and metrics. `%matplotlib inline` shows charts in the
page. Colab includes these libraries; for local use, install the project
requirements first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

print("Libraries loaded.")

## Dataset introduction

Our dataset is **house prices**, created inside the notebook with a fixed random
seed, so every run gives the same numbers.

| Column | Meaning | Units |
|---|---|---|
| `house_size_sqm` | Living area | square metres (m²) |
| `bedrooms` | Number of bedrooms | count |
| `house_age_years` | Age of the house | years |
| `price_thousand_thb` | Selling price | thousands of Thai baht (1 unit = 1,000 THB) |

The **target** is `price_thousand_thb`. The **features** are the other columns.
A price of `2000` means 2,000,000 THB (two million baht).

The data is made up for teaching. It cannot prove that size, bedrooms or age
cause a price.

In [ ]:
# Create a small house-price dataset with a fixed seed.
np.random.seed(42)

number_of_houses = 80
house_size_sqm = np.random.uniform(40, 200, number_of_houses)
bedrooms = np.random.randint(1, 5, number_of_houses)
house_age_years = np.random.randint(0, 31, number_of_houses)

# The true pattern is curved in size, with a little random noise.
noise = np.random.normal(0, 80, number_of_houses)
price_thousand_thb = (
    400
    + 6 * house_size_sqm
    + 0.08 * house_size_sqm ** 2
    + 110 * bedrooms
    - 12 * house_age_years
    + noise
)

houses = pd.DataFrame({
    "house_size_sqm": house_size_sqm,
    "bedrooms": bedrooms,
    "house_age_years": house_age_years,
    "price_thousand_thb": price_thousand_thb,
})

houses.head()

The first rows show sizes near 100 m² and prices near 1,900 thousand THB
(about 1.9 million baht). Prices run from about 687 to 4,937 thousand THB.
The term `0.08 * size²` is why a straight line will not fit perfectly.

## What regression is

**Regression** predicts a **continuous number**: a price, a temperature or a
weight. **Classification** predicts a **class label** such as "spam" or "not
spam". A target is continuous when it can take any value in a range, not only a
few groups.

**Small example.** Guessing tomorrow's temperature as `31.4` °C is regression;
guessing "hot" or "cold" is classification.

## Features and target

A **feature** is an input we measure. The **target** is the value we want to
predict. For one house:

- features: `house_size_sqm = 100`, `bedrooms = 3`, `house_age_years = 10`
- target: `price_thousand_thb = 2200` (that is 2,200,000 THB)

We write the target as `y` and the features as `x`.

## Simple linear regression

**Simple linear regression** uses one feature to predict the target with a
straight line.

**Small numerical example.** A house of 20 m² costs 1,000 thousand THB and a
house of 60 m² costs 3,000 thousand THB. The price grows by
`(3000 - 1000) / (60 - 20) = 50` per square metre, so the **slope** is `50`.
The line is `predicted = 50 * size + 0`. At 40 m² it predicts
`50 * 40 + 0 = 2000` thousand THB.

General form:

```text
predicted_price = weight * house_size + bias
```

- `weight` is also called the **slope**. Its units are target per feature, here
  **thousand THB per m²**.
- `bias` is also called the **intercept**. Its units are the target, here
  **thousand THB**.
- The pair `(weight, bias)` is written `(w, b)`.

### Finding the line by hand with OLS

**Ordinary Least Squares (OLS)** chooses the slope and intercept that make the
**squared errors** as small as possible.

**Tiny example (demonstration).** We use five clean houses. Because this is a
demonstration, all five points are used to show the calculation.

| size (m²) | price (thousand THB) |
|---|---|
| 20 | 1000 |
| 30 | 1500 |
| 40 | 2000 |
| 50 | 2500 |
| 60 | 3000 |

Step by step:

1. Mean size: `(20+30+40+50+60) / 5 = 40`.
2. Mean price: `(1000+1500+2000+2500+3000) / 5 = 2000`.
3. For each house, multiply `(size - 40)` by `(price - 2000)` and add:
   `(-20)(-1000) + (-10)(-500) + (0)(0) + (10)(500) + (20)(1000) = 50000`.
4. Add the squared size gaps:
   `400 + 100 + 0 + 100 + 400 = 1000`.
5. Slope: `50000 / 1000 = 50`.
6. Intercept: `2000 - 50 * 40 = 0`.

Formulas with symbols defined:

```text
slope     = sum((x_i - x_mean)(y_i - y_mean)) / sum((x_i - x_mean)^2)
intercept = y_mean - slope * x_mean
```

Here `x_i` is one feature value, `y_i` the matching target, `x_mean` and
`y_mean` their averages, and `sum` means "add over all examples".

In [ ]:
# Tiny demonstration data: five clean houses.
tiny_size = np.array([20.0, 30.0, 40.0, 50.0, 60.0])
tiny_price = np.array([1000.0, 1500.0, 2000.0, 2500.0, 3000.0])

# Manual OLS with the formulas above.
size_mean = tiny_size.mean()
price_mean = tiny_price.mean()
manual_slope = np.sum((tiny_size - size_mean) * (tiny_price - price_mean)) / np.sum((tiny_size - size_mean) ** 2)
manual_intercept = price_mean - manual_slope * size_mean

# scikit-learn fits the same line.
tiny_model = LinearRegression().fit(tiny_size.reshape(-1, 1), tiny_price)
sklearn_slope = tiny_model.coef_[0]
sklearn_intercept = tiny_model.intercept_

print(f"manual slope:      {manual_slope:.4f} thousand THB per m^2")
print(f"manual intercept:  {manual_intercept:.4f} thousand THB")
print(f"sklearn slope:     {sklearn_slope:.4f}")
print(f"sklearn intercept: {sklearn_intercept:.4f}")

assert np.isclose(manual_slope, 50.0)
assert np.isclose(manual_intercept, 0.0)
assert np.isclose(manual_slope, sklearn_slope)
assert np.isclose(manual_intercept, sklearn_intercept, atol=1e-9)
print("Manual OLS and sklearn agree.")

Both methods give slope `50` and intercept `0`. The slope means one extra square
metre adds 50 thousand THB to the predicted price. The intercept is the
predicted price at size zero; here it is `0`, but for real data a zero-size
house has no meaning.

### A shifted target gives a non-zero intercept

The intercept is not always zero. Add `100` to every price in the tiny example.
The mean price becomes `2000 + 100 = 2100`, and the slope stays `50` because
shifting all targets up does not change the steepness.

Expected intercept: `2100 - 50 * 40 = 100` thousand THB.

The next cell computes this intercept from the shifted data (it is not typed as a
fixed number) and checks it against scikit-learn.

In [ ]:
# Shift every tiny price up by 100 thousand THB.
shifted_price = tiny_price + 100
shifted_price_mean = shifted_price.mean()

# Compute the shifted slope and intercept from the OLS formulas.
shifted_slope = np.sum((tiny_size - size_mean) * (shifted_price - shifted_price_mean)) / np.sum((tiny_size - size_mean) ** 2)
shifted_intercept = shifted_price_mean - shifted_slope * size_mean

shifted_model = LinearRegression().fit(tiny_size.reshape(-1, 1), shifted_price)

print(f"shifted mean price:         {shifted_price_mean:.1f} thousand THB")
print(f"computed shifted slope:     {shifted_slope:.1f}")
print(f"computed shifted intercept: {shifted_intercept:.1f} thousand THB")
print(f"sklearn shifted intercept:  {shifted_model.intercept_:.1f} thousand THB")

assert np.isclose(shifted_slope, manual_slope)
assert np.isclose(shifted_intercept, 100.0)
assert np.isclose(shifted_intercept, shifted_model.intercept_)
print("The shifted intercept is 100, computed from the data and matching sklearn.")

## Actual values, predictions and residuals

The **actual** value is the real price; the **predicted** value is the model's
estimate. Error and residual measure their difference with opposite signs.

**Tiny example with both signs.** Two houses both have actual price `2000`
thousand THB.

- House A is predicted as `1950`: `error = 1950 - 2000 = -50` and
  `residual = 2000 - 1950 = +50`. The model predicted **too low**.
- House B is predicted as `2060`: `error = 2060 - 2000 = +60` and
  `residual = 2000 - 2060 = -60`. The model predicted **too high**.

General forms: `error = predicted - actual` for our gradient-descent maths;
`residual = actual - predicted` for the usual statistical convention. Thus
`residual = -error`.

A positive residual means the actual value is above the prediction. Squaring
removes the sign, so OLS and MSE can use either version; MAE uses `|error|`.

In [ ]:
# The 40 m^2 house lies on the line, so both signs are zero.
one_size = 40.0
one_actual = 2000.0
one_predicted = manual_slope * one_size + manual_intercept
one_error = one_predicted - one_actual
one_residual = one_actual - one_predicted

# Two more houses show a negative and a positive error.
low_actual = 2000.0
low_predicted = 1950.0
low_error = low_predicted - low_actual
low_residual = low_actual - low_predicted

high_actual = 2000.0
high_predicted = 2060.0
high_error = high_predicted - high_actual
high_residual = high_actual - high_predicted

print(f"exact house:   predicted {one_predicted:.1f}, actual {one_actual:.1f}, error {one_error:.1f}, residual {one_residual:.1f}")
print(f"too-low house: predicted {low_predicted:.1f}, actual {low_actual:.1f}, error {low_error:.1f}, residual {low_residual:.1f}")
print(f"too-high house: predicted {high_predicted:.1f}, actual {high_actual:.1f}, error {high_error:.1f}, residual {high_residual:.1f}")

assert one_error == 0.0
assert low_error == -50.0 and low_residual == 50.0
assert high_error == 60.0 and high_residual == -60.0
assert np.isclose(low_residual, -low_error)
assert np.isclose(high_residual, -high_error)
print("Residual is the opposite of error, and the signs behave as expected.")

### Chart: the tiny demonstration line

The chart below shows the five points, the manual OLS line and the scikit-learn
line. The two lines should sit exactly on top of each other and pass through all
five points.

In [ ]:
line_sizes = np.linspace(15, 65, 50)
manual_predictions = manual_slope * line_sizes + manual_intercept
sklearn_predictions = sklearn_slope * line_sizes + sklearn_intercept

plt.figure(figsize=(8, 5))
plt.scatter(tiny_size, tiny_price, color="tab:red", s=70, label="Actual houses")
plt.plot(line_sizes, manual_predictions, color="tab:blue", linewidth=3, label="Manual OLS line")
plt.plot(line_sizes, sklearn_predictions, color="black", linestyle="--", linewidth=2, label="sklearn line")
plt.title("Tiny demonstration: size and price")
plt.xlabel("House size (m²)")
plt.ylabel("Price (thousand THB)")
plt.legend()
plt.show()

The red points rise in a straight line. The blue manual line and the dashed black
sklearn line overlap, which confirms the hand calculation. All five points lie on
the line, so every error is zero.

## Why OLS minimises squared residuals

OLS picks the line whose **sum of squared errors** is smallest. Squaring makes
every error positive and punishes large errors more than small ones.

**Tiny check.** For the five clean houses the OLS line has zero total squared
error. Any other line must have a positive total, because these points lie
exactly on one line.

For errors `[2, -2]`, square and add: `4 + 4 = 8`. Divide by twice the
two examples: `8 / (2 * 2) = 2`. This is half the mean squared error.

The cost of a line is:

```text
cost = (1 / (2m)) * sum((predicted_i - actual_i)^2)
```

`m` is the number of examples. The `2` makes the gradient simpler later; it does
not move the best line.

In [ ]:
def squared_error_cost(actual, predicted):
    """Cost J = 1/(2m) * sum of squared errors = MSE / 2."""
    errors = predicted - actual
    return np.sum(errors ** 2) / (2 * len(actual))

ols_predictions = manual_slope * tiny_size + manual_intercept
other_predictions = 51 * tiny_size + 10

ols_cost = squared_error_cost(tiny_price, ols_predictions)
other_cost = squared_error_cost(tiny_price, other_predictions)

print(f"cost of the OLS line:   {ols_cost:.2f}")
print(f"cost of 51*size + 10:   {other_cost:.2f}")

assert np.isclose(ols_cost, 0.0)
assert other_cost > ols_cost
print("OLS has the smaller cost.")

The OLS line has cost `0.00`. The nearby line `51 * size + 10` has a larger cost,
so OLS really is the better of the two. For real data with noise, the smallest
cost is above zero.

## Simple linear regression on the house data

Now we use the real generated dataset. We keep a **test set** away from fitting.
We fit on the **training set** only, then report training and test scores
separately. We never change the model to improve the test score.

In [ ]:
# Split rows into training and test sets. random_state keeps the split fixed.
train_rows, test_rows = train_test_split(
    np.arange(len(houses)), test_size=0.2, random_state=42
)

# One feature: house size.
X_train_simple = houses.loc[train_rows, ["house_size_sqm"]]
X_test_simple = houses.loc[test_rows, ["house_size_sqm"]]
y_train = houses.loc[train_rows, "price_thousand_thb"]
y_test = houses.loc[test_rows, "price_thousand_thb"]

simple_model = LinearRegression().fit(X_train_simple, y_train)

print(f"slope (weight):   {simple_model.coef_[0]:.2f} thousand THB per m^2")
print(f"intercept (bias): {simple_model.intercept_:.2f} thousand THB")
print(f"training R-squared: {simple_model.score(X_train_simple, y_train):.4f}")
print(f"test R-squared:     {simple_model.score(X_test_simple, y_test):.4f}")

The slope is about `24.6`: one more square metre adds about 24.6 thousand THB to
the predicted price. The intercept is about `-369` thousand THB, which is
negative. That number has no real meaning because a 0 m² house cannot exist; the
line simply needs a negative intercept to sit near the data.

The training R-squared is about `0.960` and the test R-squared about `0.974`.
Both are close, so the model is not badly overfit. The test score is not used to
tune anything.

### Chart: data points and the fitted line

The chart shows training points, test points and the fitted straight line. The
line should rise with size and pass through the middle of the cloud.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train_simple["house_size_sqm"], y_train, color="tab:blue", alpha=0.7, label="Training houses")
plt.scatter(X_test_simple["house_size_sqm"], y_test, color="tab:orange", alpha=0.9, label="Test houses")
line_sizes = np.linspace(40, 200, 100)
line_prices = simple_model.coef_[0] * line_sizes + simple_model.intercept_
plt.plot(line_sizes, line_prices, color="black", linewidth=3, label="Fitted line")
plt.title("Simple linear regression: size and price")
plt.xlabel("House size (m²)")
plt.ylabel("Price (thousand THB)")
plt.legend()
plt.show()

The points rise from about 690 thousand THB near 41 m² to about 4,900 thousand
THB near 198 m². The black line follows the general upward trend. Some cheaper
houses sit below the line and some expensive houses sit above it, which is normal
because of the random noise.

### Chart: residuals of the simple model

A **residual plot** shows `residual = actual - predicted` on the vertical axis. A
good simple model should have residuals spread roughly evenly around zero.

In [ ]:
train_predictions_simple = simple_model.predict(X_train_simple)
train_residuals_simple = y_train - train_predictions_simple

plt.figure(figsize=(8, 5))
plt.scatter(X_train_simple["house_size_sqm"], train_residuals_simple, color="tab:purple", alpha=0.8)
plt.axhline(0, color="black", linewidth=2)
plt.title("Residuals of the simple model (training data)")
plt.xlabel("House size (m²)")
plt.ylabel("Residual: actual − predicted (thousand THB)")
plt.show()

The points do not sit in a flat band around zero. Middle sizes near 80-160 m²
have negative residuals, while the smallest and largest sizes have positive
residuals. This valley-shaped pattern is a sign that the true relationship is
curved, which we will handle later with a polynomial model.

## Multiple linear regression

**Multiple linear regression** uses several features at once. Each feature gets
its own weight, and there is still one bias.

**Tiny numerical example.** A 100 m², 3-bedroom, 10-year-old house, with weights
`(10, 50, -2)` and bias `100`:

1. Size part: `10 * 100 = 1000`.
2. Bedroom part: `50 * 3 = 150`.
3. Age part: `-2 * 10 = -20`.
4. Add the bias: `1000 + 150 - 20 + 100 = 1230` thousand THB.

General form:

```text
predicted_price = w_1 * x_1 + w_2 * x_2 + w_3 * x_3 + b
```

`x_1, x_2, x_3` are the feature values (size, bedrooms, age), `w_1, w_2, w_3`
their weights, and `b` the bias. Units are target per feature: thousand THB per
m², per bedroom and per year; `b` is in thousand THB.

In [ ]:
X_train_multi = houses.loc[train_rows, ["house_size_sqm", "bedrooms", "house_age_years"]]
X_test_multi = houses.loc[test_rows, ["house_size_sqm", "bedrooms", "house_age_years"]]

multi_model = LinearRegression().fit(X_train_multi, y_train)

for name, weight in zip(X_train_multi.columns, multi_model.coef_):
    print(f"weight for {name}: {weight:.2f} thousand THB per unit")
print(f"bias: {multi_model.intercept_:.2f} thousand THB")
print(f"training R-squared: {multi_model.score(X_train_multi, y_train):.4f}")
print(f"test R-squared:     {multi_model.score(X_test_multi, y_test):.4f}")

The weight for size is about `25.3`, close to the simple model. Each extra bedroom
adds about `87` thousand THB, and each extra year of age subtracts about `13`
thousand THB. These signs match the way the data was built.

The training R-squared is about `0.977` and the test R-squared about `0.983`.
Adding features improved both scores, so the extra features carry useful
information.

### Chart: predicted against actual values

The next chart plots predicted price against actual price on the **test set**.
Points near the dashed diagonal mean good predictions.

In [ ]:
test_predictions_multi = multi_model.predict(X_test_multi)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_predictions_multi, color="tab:green", s=70, label="Test houses")
limits = [
    min(y_test.min(), test_predictions_multi.min()) - 100,
    max(y_test.max(), test_predictions_multi.max()) + 100,
]
plt.plot(limits, limits, color="black", linestyle="--", linewidth=2, label="Perfect prediction")
plt.title("Multiple regression: predicted vs actual (test set)")
plt.xlabel("Actual price (thousand THB)")
plt.ylabel("Predicted price (thousand THB)")
plt.legend()
plt.show()

The points sit close to the dashed line, so predictions are near the actual
prices. No point is far from the line, which fits the high test R-squared of
about `0.983`.

## Error, loss and cost

These three words are close but not the same:

- **error**: the signed prediction difference for one example.
- **loss**: a penalty for one example; here we square its error.
- **cost**: here, half the average squared-error loss across examples.

**Tiny example.** Two houses have errors `[2, -2]`. Their squared-error losses are
`[2², (-2)²] = [4, 4]`. The mean loss (MSE) is `(4 + 4) / 2 = 4`. The cost is
`4 / 2 = 2`.

General form for `m` examples:

```text
loss_i = (predicted_i - actual_i)^2
MSE  = (1 / m) * sum(loss_i) = mean squared-error loss
cost = (1 / 2) * MSE = (1 / (2m)) * sum(loss_i)
```

`m` is the number of examples, `loss_i = (predicted_i - actual_i)²` for example
`i`, `predicted_i` the model output, `actual_i` the true target, and `sum` means
"add over all examples". The extra `1/2` makes the gradient simpler later and
does not move the best line.

## Mean Absolute Error (MAE)

MAE is the average size of the errors, ignoring sign.

**Tiny example.** Actual `[10, 20, 30]`, predicted `[12, 18, 33]`. Errors are
`[2, -2, 3]`. Absolute errors are `[2, 2, 3]`. MAE is `(2 + 2 + 3) / 3 = 2.333`
thousand THB.

```text
MAE = (1 / m) * sum(|predicted_i - actual_i|)
```

- Units: the same as the target.
- Easy to explain: "on average the prediction is off by 2.333".
- Treats all errors in a straight line; a few large errors do not dominate.

## Mean Squared Error (MSE)

MSE is the average of the squared errors. It punishes large errors more.

**Tiny example.** With the same numbers, squared errors are `[4, 4, 9]`. MSE is
`(4 + 4 + 9) / 3 = 5.667`.

```text
MSE = (1 / m) * sum((predicted_i - actual_i)^2)
```

- Units: target squared (thousand THB squared), so it is hard to read directly.
- Large errors get much more weight than small errors.

## Root Mean Squared Error (RMSE)

RMSE is the square root of MSE, which brings the units back to the target.

**Tiny example.** `sqrt(5.667) = 2.380` thousand THB.

```text
RMSE = sqrt( (1 / m) * sum((predicted_i - actual_i)^2) )
```

- Units: the same as the target.
- Still punishes large errors, but the number is easy to compare with prices.

## R-squared

R-squared measures the share of the target's variation that the model explains.
It compares the model with a simple baseline that always predicts the mean.

**Tiny example.** Actual `[10, 20, 30]` has mean `20`. The total variation is
`(10-20)² + (20-20)² + (30-20)² = 200`. The model's squared error is `17`. So
R-squared is `1 - 17 / 200 = 0.915`. The model explains about 91.5% of the
variation.

```text
R-squared = 1 - (sum of squared errors) / (sum of squared gaps from the mean)
```

- `1.0` is perfect, `0.0` is no better than the mean.
- It can be **negative** if the model is worse than always predicting the mean.

## Choosing a metric

- Use **MAE** when you want an average error in target units and easy wording.
- Use **RMSE** when large errors are especially bad.
- Use **MSE** mainly inside training and maths, because its units are squared.
- Use **R-squared** to describe the share of variation explained, not the size of
  a typical error.

**Negative R-squared.** A model can be worse than the mean baseline. For actual
`[10, 20, 30]` and predicted `[0, 0, 0]`, the squared error is
`100 + 400 + 900 = 1400`, so R-squared is `1 - 1400 / 200 = -6.0`. A negative
value is a warning sign.

**Constant target.** If every actual value is the same, the gaps from the mean are
all zero. Then R-squared divides by zero and has no meaning. Use MAE or RMSE
instead.

In [ ]:
def mae(actual, predicted):
    return np.mean(np.abs(predicted - actual))

def mse(actual, predicted):
    return np.mean((predicted - actual) ** 2)

def rmse(actual, predicted):
    return np.sqrt(mse(actual, predicted))

def r_squared(actual, predicted):
    total_variation = np.sum((actual - np.mean(actual)) ** 2)
    squared_error = np.sum((predicted - actual) ** 2)
    return 1 - squared_error / total_variation

tiny_actual = np.array([10.0, 20.0, 30.0])
tiny_predicted = np.array([12.0, 18.0, 33.0])

print(f"MAE:  {mae(tiny_actual, tiny_predicted):.3f}")
print(f"MSE:  {mse(tiny_actual, tiny_predicted):.3f}")
print(f"RMSE: {rmse(tiny_actual, tiny_predicted):.3f}")
print(f"R-squared: {r_squared(tiny_actual, tiny_predicted):.3f}")

assert np.isclose(mae(tiny_actual, tiny_predicted), 7 / 3)
assert np.isclose(mse(tiny_actual, tiny_predicted), 17 / 3)
assert np.isclose(rmse(tiny_actual, tiny_predicted), np.sqrt(17 / 3))
assert np.isclose(r_squared(tiny_actual, tiny_predicted), 1 - 17 / 200)
print("The hand calculations and the functions agree.")

The printed values are `2.333`, `5.667`, `2.380` and `0.915`. They match the hand
calculations in the notes above, so the functions are correct.

### Metrics on the test set

Now we apply the functions to the held-out test set and compare them with
scikit-learn. We report training and test scores separately.

In [ ]:
def score_model(name, model, X_train_data, X_test_data, y_train_data, y_test_data):
    train_predictions = model.predict(X_train_data)
    test_predictions = model.predict(X_test_data)
    print(f"--- {name} ---")
    print(f"training MAE:  {mae(y_train_data, train_predictions):.2f}")
    print(f"training RMSE: {rmse(y_train_data, train_predictions):.2f}")
    print(f"training R-squared: {r_squared(y_train_data, train_predictions):.4f}")
    print(f"test MAE:      {mae(y_test_data, test_predictions):.2f}")
    print(f"test RMSE:     {rmse(y_test_data, test_predictions):.2f}")
    print(f"test R-squared: {r_squared(y_test_data, test_predictions):.4f}")
    print(f"sklearn test MAE: {mean_absolute_error(y_test_data, test_predictions):.2f}")
    print(f"sklearn test RMSE: {np.sqrt(mean_squared_error(y_test_data, test_predictions)):.2f}")
    print(f"sklearn test R-squared: {r2_score(y_test_data, test_predictions):.4f}")

score_model("simple model", simple_model, X_train_simple, X_test_simple, y_train, y_test)
score_model("multiple model", multi_model, X_train_multi, X_test_multi, y_train, y_test)

For the simple model the test MAE is about `161` and the test RMSE about `193`
thousand THB, with test R-squared about `0.974`. RMSE is larger than MAE, which
is normal because RMSE punishes the largest errors. The multiple model has a
smaller test MAE (`134`) and test RMSE (`155`), and a higher test R-squared
(`0.983`). The hand-made functions and scikit-learn print the same values.

In [ ]:
bad_predictions = np.zeros_like(tiny_actual)
constant_actual = np.array([5.0, 5.0, 5.0])

print(f"R-squared for always-zero predictions: {r_squared(tiny_actual, bad_predictions):.3f}")
print(f"total variation for a constant target: {np.sum((constant_actual - constant_actual.mean()) ** 2):.1f}")

assert r_squared(tiny_actual, bad_predictions) < 0
assert np.isclose(np.sum((constant_actual - constant_actual.mean()) ** 2), 0.0)
print("Negative R-squared and a zero denominator are confirmed.")

The always-zero model has R-squared `-6.0`, which is worse than the mean
baseline. The constant target has total variation `0.0`, so the R-squared formula
would divide by zero; MAE or RMSE should be used instead.

## Gradient descent

OLS has a direct formula, but many models have no direct formula. **Gradient
descent** finds good parameters by starting from a guess and taking small steps
downhill on the cost surface.

**Tiny example.** On a hill, a large slope means a big step and a small slope
means a small step. The step size is also controlled by the **learning rate**
`alpha`. If `alpha` is too large, the steps overshoot and the cost grows.

### Worked numerical step

Use the five clean houses (`x` from 20 to 60, `y` from 1000 to 3000), start with
`w = 0`, `b = 0`, and learning rate `alpha = 0.0001`.

1. Predictions are all `0`, so errors `predicted - actual` are
   `-1000, -1500, -2000, -2500, -3000`.
2. For the weight, multiply each error by its size, add, and divide by 5:
   `(-1000*20 - 1500*30 - 2000*40 - 2500*50 - 3000*60) / 5 = -90000`.
3. For the bias, add the errors and divide by 5: `(-10000) / 5 = -2000`.
4. Move against each number by `alpha`:
   new `w = 0 - 0.0001 * (-90000) = 9` and new `b = 0 - 0.0001 * (-2000) = 0.2`.

The cost falls from `2,250,000` to about `1,512,572`, so the step went downhill.

### Weight and bias updates

The two numbers above have names. The gradient of the cost with respect to the
weight and the bias is:

```text
dJ/dw = (1 / m) * sum(error_i * x_i)
dJ/db = (1 / m) * sum(error_i)
```

`m` is the number of examples, `error_i = predicted_i - actual_i`, `x_i` is the
feature of example `i`, and `sum` adds over all examples. The update rule moves
each parameter **against** the gradient:

```text
w = w - alpha * dJ/dw
b = b - alpha * dJ/db
```

`alpha` is the learning rate. Both updates use the old `w` and `b` at the same
time (**simultaneous** update).

In [ ]:
def compute_cost(x, y, weight, bias):
    """Squared-error cost J(w, b) = 1/(2m) * sum of squared errors."""
    predictions = weight * x + bias
    errors = predictions - y
    return np.sum(errors ** 2) / (2 * len(y))

def compute_gradient(x, y, weight, bias):
    """Gradients of the cost with respect to weight and bias."""
    predictions = weight * x + bias
    errors = predictions - y
    dJ_dw = np.sum(errors * x) / len(y)
    dJ_db = np.sum(errors) / len(y)
    return dJ_dw, dJ_db

def gradient_descent(x, y, weight_start, bias_start, learning_rate, number_of_steps):
    """Run batch gradient descent and record the cost at every step."""
    weight = weight_start
    bias = bias_start
    cost_history = [compute_cost(x, y, weight, bias)]
    for _ in range(number_of_steps):
        dJ_dw, dJ_db = compute_gradient(x, y, weight, bias)
        weight = weight - learning_rate * dJ_dw
        bias = bias - learning_rate * dJ_db
        cost_history.append(compute_cost(x, y, weight, bias))
    return weight, bias, np.array(cost_history)

In [ ]:
# Check the hand-worked step in code.
dJ_dw_step, dJ_db_step = compute_gradient(tiny_size, tiny_price, 0.0, 0.0)
print(f"hand dJ/dw: {dJ_dw_step:.0f}, hand dJ/db: {dJ_db_step:.0f}")
assert np.isclose(dJ_dw_step, -90000)
assert np.isclose(dJ_db_step, -2000)

# Standardize the feature so gradient descent converges in fewer steps.
size_mean_train = tiny_size.mean()
size_std_train = tiny_size.std()
tiny_size_scaled = (tiny_size - size_mean_train) / size_std_train

weight_scaled, bias_scaled, cost_history = gradient_descent(
    tiny_size_scaled, tiny_price, 0.0, 0.0, learning_rate=0.1, number_of_steps=2000
)

# Convert the scaled parameters back to the original units.
slope_from_gd = weight_scaled / size_std_train
intercept_from_gd = bias_scaled - weight_scaled * size_mean_train / size_std_train

print(f"gradient-descent slope:     {slope_from_gd:.4f} thousand THB per m^2")
print(f"gradient-descent intercept: {intercept_from_gd:.4f} thousand THB")
print(f"cost at start: {cost_history[0]:.2f}")
print(f"cost at end:   {cost_history[-1]:.2f}")

assert np.all(np.isfinite(cost_history))
assert np.all(np.diff(cost_history) <= 1e-6)
assert cost_history[-1] < cost_history[0]
assert np.isclose(slope_from_gd, manual_slope, atol=1e-6)
assert np.isclose(intercept_from_gd, manual_intercept, atol=1e-6)
print("Cost is finite and decreasing, and gradient descent reached the OLS line.")

The hand gradient values `-90000` and `-2000` match the code exactly. After 2000
steps the scaled parameters convert back to slope `50` and intercept `0`, the same
line that OLS found. The cost is finite at every step and never increases after
the first few steps, so gradient descent converged.

### Chart: cost against training iterations

The chart has two panels. The left panel uses the good learning rate `0.1`; the
cost should fall quickly and then flatten. The right panel uses a large learning
rate `2.1`; the cost should grow instead of fall.

In [ ]:
_, _, diverging_history = gradient_descent(
    tiny_size_scaled, tiny_price, 0.0, 0.0, learning_rate=2.1, number_of_steps=30
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cost_history, color="tab:blue")
axes[0].set_title("Good learning rate = 0.1")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Cost")
axes[1].plot(diverging_history, color="tab:red")
axes[1].set_title("Learning rate too large = 2.1")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Cost")
fig.suptitle("Cost history of gradient descent")
plt.tight_layout()
plt.show()

In the left panel the cost drops from about 2,250,000 to nearly 0 and then stays
flat, which means the steps reached the bottom of the cost bowl. In the right
panel the cost grows very fast and reaches huge values, which means the learning
rate is too large and the steps overshoot. A good learning rate makes the cost
fall and settle; a bad one makes it grow or jump around.

## Polynomial regression and nonlinear relationships

A straight line cannot follow a curve. **Polynomial regression** adds powers of a
feature, for example `size` and `size²`, and then uses the same linear regression
machinery.

**Tiny numerical example.** With weights `w_1 = 5`, `w_2 = 0.1` and bias `b = 500`:

1. At size `50`: `5*50 + 0.1*50² + 500 = 250 + 250 + 500 = 1000` thousand THB.
2. At size `150`: `5*150 + 0.1*150² + 500 = 750 + 2250 + 500 = 3500` thousand THB.

From size 50 to 150 the linear part grows by `500`, but the squared part grows by
`2000`, so the curve bends upward.

General form:

```text
predicted_price = w_1 * size + w_2 * size^2 + b
```

`w_1` is the weight for size, `w_2` the weight for the squared size, and `b` the
bias. We choose the **degree before looking at the test set** and keep it fixed
at **degree 2** (a quadratic). A higher degree can bend more, but with few
examples it can also learn the noise.

In [ ]:
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
)
poly_model.fit(X_train_simple, y_train)

poly_weights = poly_model.named_steps["linearregression"].coef_
poly_bias = poly_model.named_steps["linearregression"].intercept_
print(f"weight for size:   {poly_weights[0]:.2f}")
print(f"weight for size^2: {poly_weights[1]:.4f}")
print(f"bias: {poly_bias:.2f}")
print(f"training R-squared: {poly_model.score(X_train_simple, y_train):.4f}")
print(f"test R-squared:     {poly_model.score(X_test_simple, y_test):.4f}")

The `size²` weight is positive (`0.0796`), so the curve bends upward. The training
R-squared rises to about `0.976` and the test R-squared to about `0.980`, both
higher than the straight line. The curve fits the real shape better without being
tuned on the test set.

### Chart: straight line against polynomial curve

The chart shows the training points, the straight line and the degree-2 curve.
The curve should bend upward and follow the points at large sizes better than the
straight line.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train_simple["house_size_sqm"], y_train, color="tab:blue", alpha=0.6, label="Training houses")
plt.plot(line_sizes, simple_model.coef_[0] * line_sizes + simple_model.intercept_, color="black", linewidth=3, label="Straight line (degree 1)")
poly_line_sizes = pd.DataFrame({"house_size_sqm": np.linspace(40, 200, 200)})
plt.plot(poly_line_sizes["house_size_sqm"], poly_model.predict(poly_line_sizes), color="tab:red", linewidth=3, label="Polynomial (degree 2)")
plt.title("Straight line vs polynomial curve")
plt.xlabel("House size (m²)")
plt.ylabel("Price (thousand THB)")
plt.legend()
plt.show()

The black straight line is a good first summary. The red curve bends upward and
passes closer to the large, expensive houses. The gap between the two lines is
small at 40 m² and wider near 200 m², which matches the `size²` term.

## Common mistakes

- **Using regression for labels.** A yes/no or class target needs classification,
  not regression.
- **Mixing up error signs.** Here `error = predicted - actual` everywhere. Pick
  one rule and keep it.
- **Forgetting units.** A slope of `50` means 50 thousand THB per m², not per
  house.
- **Thinking the intercept always makes sense.** A negative intercept is fine for
  the maths even when size zero is impossible.
- **Judging a model on training data only.** Always keep a test set and report its
  score separately.
- **Tuning on the test set.** Choose settings such as the polynomial degree
  before looking at the test score.
- **Reading R-squared alone.** It ignores the size of a typical error; also report
  MAE or RMSE.
- **Trusting a negative R-squared.** It means the model is worse than predicting
  the mean.
- **Dividing by zero in R-squared.** A constant target makes the denominator zero;
  use MAE or RMSE.
- **Using a huge learning rate.** Cost that grows means `alpha` is too large.
- **Choosing a very high polynomial degree.** It can fit noise and fail on new
  data.

## Summary

In this notebook we learned how regression predicts continuous numbers:

- **Simple linear regression** uses one feature:
  `predicted = weight * feature + bias`.
- **OLS** finds the slope and intercept that minimise squared errors, both by hand
  and with scikit-learn `LinearRegression`.
- **Multiple linear regression** adds more features, each with its own weight.
- **Error**, **loss** and **cost** describe one example, one penalty and half the
  average penalty.
- **MAE**, **MSE**, **RMSE** and **R-squared** each answer a different question;
  R-squared can be negative and fails for a constant target.
- **Gradient descent** reduces the cost step by step; the learning rate controls
  the step size and the cost history shows convergence.
- **Polynomial regression** fits a curved relationship with a fixed degree.